In [16]:
import sounddevice as sd
import numpy as np
import queue
import threading
import logging
import joblib
import librosa
import os
import sys
from datetime import datetime

In [17]:
SAMPLE_RATE          = 22050   # must match training
CHUNK_SECONDS        = 2       # seconds per inference window
CHUNK_SAMPLES        = SAMPLE_RATE * CHUNK_SECONDS
ANGER_LABEL          = "ANG"   # CREMA-D anger label
HAPPINESS_LABEL = "HAP"
SADNESS_LABEL = "SAD"
CONFIDENCE_THRESHOLD = 0.60    # only log if confidence >= this
OVERLAP              = 0.5     # 50% overlap for file mode (0.0 to disable)
MODEL_PATH           = "emotion_model.pkl"
LOG_FILE             = "anger_log.txt"

In [18]:
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s — %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
 
def log_anger(confidence=None, source="mic"):
    msg = f"ANGER DETECTED | source={source}"
    if confidence is not None:
        msg += f" | confidence={confidence:.0%}"
    logging.info(msg)
    print(f"\n  ⚠  {msg}\n")

def log_emotion(emotion, confidence=None, source="mic"):
    msg = f"{emotion.upper()} DETECTED | source ={source}"
    if confidence is not None:
        msg += f" | confidence={confidence:.0%}"
        logging.info(msg)
        print(f"\n  ⚠  {msg}\n")

In [19]:
def load_model(path):
    if not os.path.exists(path):
        print(f"[ERROR] Model file not found: {path}")
        sys.exit(1)
    model = joblib.load(path)
    print(f"[OK] Model loaded from '{path}'")
    return model

In [20]:
def extract_features(audio_chunk):
    X = audio_chunk.astype(np.float32)
 
    # Normalize (same as training)
    X = librosa.util.normalize(X)
 
    # Trim silence — top_db=20 is gentler than default 60
    # (avoids over-trimming on live mic audio)
    X, _ = librosa.effects.trim(X, top_db=20)
 
    # Safety guard: pad if chunk is too short after trimming
    if len(X) < 512:
        X = np.pad(X, (0, 512 - len(X)))
 
    result = np.array([])
 
    # 1. Chroma STFT — 12 features
    stft   = np.abs(librosa.stft(X))
    chroma = np.mean(
        librosa.feature.chroma_stft(S=stft, sr=SAMPLE_RATE).T,
        axis=0
    )
    result = np.hstack((result, chroma))
 
    # 2. MFCCs — 40 features
    mfccs = np.mean(
        librosa.feature.mfcc(y=X, sr=SAMPLE_RATE, n_mfcc=40).T,
        axis=0
    )
    result = np.hstack((result, mfccs))
 
    # 3. Mel Spectrogram — 128 features
    mel = np.mean(
        librosa.feature.melspectrogram(y=X, sr=SAMPLE_RATE).T,
        axis=0
    )
    result = np.hstack((result, mel))
 
    # Final shape: (1, 180)
    return result.reshape(1, -1)

In [21]:
def run_inference(model, chunk, source="mic"):
    try:
        features    = extract_features(chunk)
        prediction  = model.predict(features)[0]
        confidence  = None
 
        if hasattr(model, "predict_proba"):
            proba      = model.predict_proba(features)[0]
            confidence = float(max(proba))
 
        label = f"{prediction}"
        if confidence is not None:
            label += f" ({confidence:.0%})"
        print(f"  [{datetime.now().strftime('%H:%M:%S')}] Detected: {label}")
 
        if prediction == ANGER_LABEL and (confidence is None or confidence >= CONFIDENCE_THRESHOLD):
            log_emotion("ANGER",confidence, source=source)
        elif prediction == HAPPINESS_LABEL and (confidence is None or confidence >= 0.70):  # Custom threshold
            log_emotion("HAPPINESS", confidence, source=source)
        elif prediction == SADNESS_LABEL and (confidence is None or confidence >= 0.70):  # Custom threshold
            log_emotion("SADNESS", confidence, source=source)
 
    except Exception as e:
        print(f"  [inference error] {e}")

In [22]:
audio_queue = queue.Queue()

In [23]:
def consumer_thread(model, source="mic"):
    print("[thread] Consumer started — processing audio chunks...")
    while True:
        item = audio_queue.get()
 
        if item is None:          # poison pill → shutdown signal
            print("[thread] Consumer shutting down.")
            break
 
        chunk, src = item
        run_inference(model, chunk, source=src)

In [24]:
def audio_callback(indata, frames, time_info, status):
    """Runs on sounddevice's audio thread for every chunk."""
    if status:
        print(f"  [stream] {status}")
    # indata shape: (frames, channels) — take mono channel
    audio_queue.put((indata[:, 0].copy(), "mic"))
 
 
def run_microphone(model):
    worker = threading.Thread(
        target=consumer_thread,
        args=(model, "mic"),
        daemon=True
    )
    worker.start()
 
    print(f"\n[MIC] Streaming at {SAMPLE_RATE} Hz | chunk={CHUNK_SECONDS}s")
    print("      Press Ctrl+C to stop.\n")
 
    with sd.InputStream(
        samplerate=SAMPLE_RATE,
        channels=1,
        dtype="float32",
        blocksize=CHUNK_SAMPLES,
        callback=audio_callback
    ):
        try:
            while True:
                sd.sleep(500)
        except KeyboardInterrupt:
            print("\n[MIC] Stopping...")
 
    audio_queue.put(None)    # signal consumer to exit
    worker.join()
    print("[MIC] Done. Anger events saved to:", LOG_FILE)

In [25]:
def run_from_file(model, filepath):
    if not os.path.exists(filepath):
        print(f"[ERROR] File not found: {filepath}")
        return
 
    worker = threading.Thread(
        target=consumer_thread,
        args=(model, filepath),
        daemon=True
    )
    worker.start()
 
    print(f"\n[FILE] Loading: {filepath}")
    audio, sr = librosa.load(filepath, sr=SAMPLE_RATE, mono=True)
    print(f"[FILE] Duration: {len(audio)/SAMPLE_RATE:.1f}s | "
          f"Sample rate: {sr} Hz\n")
 
    step = int(CHUNK_SAMPLES * (1.0 - OVERLAP))   # hop size
    total_chunks = max(1, (len(audio) - CHUNK_SAMPLES) // step + 1)
    processed    = 0
 
    for start in range(0, len(audio) - CHUNK_SAMPLES + 1, step):
        chunk = audio[start : start + CHUNK_SAMPLES]
        audio_queue.put((chunk, filepath))
        processed += 1
        pct = int(processed / total_chunks * 100)
        print(f"  [file] chunk {processed}/{total_chunks} "
              f"@ {start/SAMPLE_RATE:.1f}s ({pct}%)", end="\r")
 
    audio_queue.put(None)    # signal consumer to exit
    worker.join()
    print(f"\n[FILE] Done. Processed {processed} chunks.")
    print("[FILE] Anger events saved to:", LOG_FILE)

In [26]:
if __name__ == "__main__":
    model = load_model(MODEL_PATH)
 
    # ── Choose mode ──────────────────────────
    # For microphone:
    run_microphone(model)

c:\Users\VICTUS\miniconda3\envs\audioPipeline\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelBinarizer from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\VICTUS\miniconda3\envs\audioPipeline\lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator MLPClassifier from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


[OK] Model loaded from 'emotion_model.pkl'
[thread] Consumer started — processing audio chunks...

[MIC] Streaming at 22050 Hz | chunk=2s
      Press Ctrl+C to stop.

  [14:40:29] Detected: FEA (100%)
  [14:40:31] Detected: HAP (100%)

  ⚠  HAPPINESS DETECTED | source =mic | confidence=100%

  [14:40:33] Detected: HAP (100%)

  ⚠  HAPPINESS DETECTED | source =mic | confidence=100%

  [14:40:35] Detected: FEA (100%)
  [14:40:37] Detected: FEA (100%)
  [14:40:39] Detected: FEA (100%)
  [14:40:41] Detected: FEA (100%)
  [14:40:43] Detected: HAP (100%)

  ⚠  HAPPINESS DETECTED | source =mic | confidence=100%

  [14:40:45] Detected: HAP (100%)

  ⚠  HAPPINESS DETECTED | source =mic | confidence=100%

  [14:40:47] Detected: HAP (100%)

  ⚠  HAPPINESS DETECTED | source =mic | confidence=100%

  [14:40:49] Detected: FEA (100%)
  [14:40:51] Detected: FEA (100%)
  [14:40:53] Detected: FEA (100%)
  [14:40:55] Detected: FEA (100%)
  [14:40:57] Detected: FEA (100%)
  [14:40:59] Detected: FEA (100%)